In [0]:

%pip install geopy requests sentence-transformers trafilatura

In [0]:
%restart_python

In [0]:
dbutils.widgets.text("nws_api_base_url", "https://api.weather.gov", "NWS API base URL")
dbutils.widgets.text("alert_table_name", "weather_alert_documents", "WEATHER ALERT TABLE")
dbutils.widgets.text("embedding_model", "sentence-transformers/all-MiniLM-L6-v2", "Embedding model")
dbutils.widgets.text("embeddings_table_name", "weather_alert_embeddings", "Alert Embeddings")
dbutils.widgets.text("chunk_size", "800", "Alert content chunk size (chars)")
dbutils.widgets.text("chunk_overlap", "100", "Alert content chunk overlap (chars)")


NWS_API_BASE_URL = dbutils.widgets.get("nws_api_base_url")
ALERT_TABLE_NAME = dbutils.widgets.get("alert_table_name")
EMBEDDING_MODEL_NAME = dbutils.widgets.get("embedding_model")
EMBEDDINGS_TABLE_NAME = dbutils.widgets.get("embeddings_table_name")
CHUNK_SIZE = int(dbutils.widgets.get("chunk_size"))
CHUNK_OVERLAP = int(dbutils.widgets.get("chunk_overlap"))

In [0]:
import base64
from urllib.parse import urlparse

from databricks.sdk import WorkspaceClient

w = WorkspaceClient()


def get_lakebase_url() -> str:
    secret = w.secrets.get_secret(scope="database", key="lakebase-url")
    return base64.b64decode(secret.value).decode("utf-8")


lakebase_url = get_lakebase_url()
parsed = urlparse(lakebase_url)

# Extract connection details directly from the secret URL
db_host = parsed.hostname
db_port = parsed.port or 5432
db_name = parsed.path.lstrip('/')
db_user = parsed.username
db_password = parsed.password

print(f"Connection details:")
print(f"  Host: {db_host}:{db_port}")
print(f"  Database: {db_name}")
print(f"  User: {db_user}")
print(f"  Using raw credentials from secret (no OAuth)")

In [0]:
import geopy
from geopy.geocoders import Nominatim
import requests
import json as _json

geolocator = Nominatim(user_agent="my_coordinate_finder")

# def fetch_forecast_endpoint(session: requests.Session, longitude: float, latitude: float) -> list[dict]:
#     """Single GET /v2/reference/news call for one ticker (mirrors
#     MassiveClient.get_news in massive_client.py)."""
#     resp = session.get(
#         f"{NWS_API_BASE_URL}/points/{latitude},{longitude}"
#     )
#     resp.raise_for_status()
#     return resp.json().get("properties", {}).get("forecast")

def fetch_active_alerts(session:requests.Session, state:str):
    response = session.get(
        f"{NWS_API_BASE_URL}/alerts/active",
        params={"area":state}
    )
    response.raise_for_status()
    # Return all features' properties, not just the first one
    return [feature.get("properties", {}) for feature in response.json().get("features", [])]

def sync_with_lakebase_alert_table(city:str, alerts:list[dict]):
    if not alerts:
        return 0
    rows = []
    for alert in alerts:
        rows.append({
            "id":str(alert.get("id")),
            "location": city,
            "source_type": "alert",
            "headline": alert.get("headline"),
            "description": alert.get("description"),
            "instruction": alert.get("instruction"),
            "issued_at": alert.get("sent"),
            "payload": _json.dumps(alert)
        })
    return rows


# def get_forecast_for_city(session: requests.Session, forecast_endpoint:str):
#     reponse = session.get(forecast_endpoint)
#     reponse.raise_for_status()
#     return reponse.json().get("properties",{}).get("periods", [])

location_list = ["Boston, MA", "Austin, TX", "New York, NY", "Denver, CO", "San Francisco, CA"]
# , "Austin, TX", "New York, NY", "Denver, CO", "San Francisco, CA"
print(f"{len(location_list)} locations found")
_nws_session = requests.Session()
all_news_rows = []
try:
    for city in location_list:
        location = geolocator.geocode(city)
        if location:
            print(f"City Name:{city}")
            print(f"Latitude:{round(location.latitude,4)}")
            print(f"Longitude:{round(location.longitude,4)}")
            # forecast_endpoint = fetch_forecast_endpoint(_nws_session, round(location.longitude,4), round(location.latitude,4))
            # forecast = get_forecast_for_city(_nws_session, forecast_endpoint)
            # print(f"forecast: {forecast}")
            alerts = fetch_active_alerts(_nws_session, city[-2:])
            batch_rows = sync_with_lakebase_alert_table(city, alerts)
            if batch_rows:
                all_news_rows.extend(batch_rows)
            print(f"alert:{alerts}")
            print(f"\nCollected {len(all_news_rows)} alerts to insert via Lakebase SDK.")
        else:
            print("Location Not found")
    # for i in all_news_rows:
    #     print(i)
except Exception as e:
    print(f"Error is: {e}")
        







In [0]:
import psycopg2

# Build connection using psycopg2
conn = psycopg2.connect(
    host=db_host,
    port=db_port,
    dbname=db_name,
    user=db_user,
    password=db_password,
    sslmode='require'
)

try:
    cursor = conn.cursor()
    
    # Prepare data tuples for batch insert
    insert_data = [
        (
            row['id'],
            row['location'],
            row['source_type'],
            row['headline'],
            row['description'],
            row['instruction'],
            row['issued_at'],
            row['payload']
        )
        for row in all_news_rows
    ]
    
    # Batch insert with ON CONFLICT DO NOTHING for deduplication
    insert_sql = f"""
        INSERT INTO {ALERT_TABLE_NAME} (
            id, location, source_type, headline, description, instruction, issued_at,
            payload, synced_at
        ) VALUES (%s, %s, %s, %s, %s, %s, %s, %s, CURRENT_TIMESTAMP)
        ON CONFLICT (id) DO NOTHING
    """
    
    # executemany in psycopg3 is much faster than individual INSERTs
    cursor.executemany(insert_sql, insert_data)
    
    conn.commit()
    inserted_count = cursor.rowcount
    print(f"✅ Successfully inserted {inserted_count} new alerts")
    print(f"   (Duplicates were skipped via ON CONFLICT DO NOTHING)")
    
finally:
    cursor.close()
    conn.close()

In [0]:
import pandas as pd
import psycopg2

# Load news documents using psycopg2
conn = psycopg2.connect(
    host=db_host,
    port=db_port,
    dbname=db_name,
    user=db_user,
    password=db_password,
    sslmode='require'
)

try:
    # Query with embedding_text computed
    query = f"""
        SELECT 
            id,
            location,
            source_type,
            headline,
            issued_at,
            TRIM(CONCAT(COALESCE(description, ''), '. ', COALESCE(instruction, ''))) AS embedding_text
        FROM {ALERT_TABLE_NAME}
        WHERE TRIM(CONCAT(COALESCE(location, ''), '. ', COALESCE(description, ''))) IS NOT NULL
          AND TRIM(CONCAT(COALESCE(location, ''), '. ', COALESCE(description, ''))) != ''
    """
    
    alerts_df = pd.read_sql_query(query, conn)
    print(f"Loaded {len(alerts_df)} news documents from {ALERT_TABLE_NAME}")
    display(alerts_df.head(5))
finally:
    conn.close()

In [0]:
import pandas as pd
import requests
import trafilatura


print(f"Fetching and chunking content from {len(alerts_df)} alert texts...")

# Fetch and chunk alert content
out_alert_ids, out_locations, out_chunk_indexes, out_chunk_texts = [], [], [], []

for idx, row in alerts_df.iterrows():
    alert_id = row['id']
    location = row['location']
    narrative_text = row['embedding_text']
    
    # try:
    #     resp = requests.get(headline, timeout=15)
    #     resp.raise_for_status()
    #     text = trafilatura.extract(resp.text)
    # except Exception as e:
    #     # Dead link, paywall, timeout, etc. - skip this article's
    #     # content chunks rather than failing the whole job.
    #     continue

    # if not text:
    #     continue

    # Split into overlapping chunks
    for chunk_index, start in enumerate(range(0, len(narrative_text), CHUNK_SIZE - CHUNK_OVERLAP)):
        chunk_text = narrative_text[start : start + CHUNK_SIZE].strip()
        if not chunk_text:
            continue
        out_alert_ids.append(alert_id)
        out_locations.append(location)
        out_chunk_indexes.append(str(chunk_index))
        out_chunk_texts.append(chunk_text)
        if start + CHUNK_SIZE >= len(narrative_text):
            break
    
    # Progress update every 10 articles
    if (idx + 1) % 10 == 0:
        print(f"  Processed {idx + 1}/{len(alerts_df)} articles")

chunks_df = pd.DataFrame({
    "article_id": out_alert_ids,
    "ticker": out_locations,
    "chunk_index": out_chunk_indexes,
    "chunk_text": out_chunk_texts,
})

print(f"Extracted {len(chunks_df)} content chunks from {len(alerts_df)} article URLs")
display(chunks_df.head(5))

In [0]:
import os
import pandas as pd
from sentence_transformers import SentenceTransformer

# Model should already be loaded from earlier, but ensure cache is set
os.environ["HF_HOME"] = "/tmp/.cache/huggingface"
os.environ["TRANSFORMERS_CACHE"] = "/tmp/.cache/huggingface"
os.environ["HF_HUB_CACHE"] = "/tmp/.cache/huggingface"

print(f"Computing chunk embeddings using {EMBEDDING_MODEL_NAME}...")
# Reuse the model if already loaded, otherwise load it
if 'model' not in locals():
    print("Loading embedding model...")
    model = SentenceTransformer(EMBEDDING_MODEL_NAME, cache_folder="/tmp/.cache/huggingface")

# Compute chunk embeddings in batches
batch_size = 32
all_chunk_embeddings = []

for i in range(0, len(chunks_df), batch_size):
    batch = chunks_df.iloc[i:i+batch_size]
    vectors = model.encode(batch["chunk_text"].tolist(), show_progress_bar=False)
    all_chunk_embeddings.extend(vectors.tolist())
    if (i + batch_size) % 128 == 0:
        print(f"  Processed {min(i + batch_size, len(chunks_df))}/{len(chunks_df)} chunks")

# Create chunk embeddings DataFrame
chunk_embeddings_df = pd.DataFrame({
    "alert_id": chunks_df["article_id"],
    "location": chunks_df["ticker"],
    "chunk_index": chunks_df["chunk_index"],
    "chunk_text": chunks_df["chunk_text"],
    "embedding": all_chunk_embeddings,
})

print(f"Computed {len(chunk_embeddings_df)} chunk embeddings using {EMBEDDING_MODEL_NAME}")

In [0]:
import psycopg2
from datetime import datetime

# Add id (article_id_chunk_index), model_name, and embedded_at columns
chunk_embeddings_df['id'] = chunk_embeddings_df['alert_id'] + '_' + chunk_embeddings_df['chunk_index']
chunk_embeddings_df['model_name'] = EMBEDDING_MODEL_NAME
chunk_embeddings_df['embedded_at'] = datetime.now()
chunk_embeddings_df['chunk_index'] = chunk_embeddings_df['chunk_index'].astype(int)

chunk_embeddings_rows = chunk_embeddings_df.to_dict('records')

if len(chunk_embeddings_rows) > 0:
    print(f"Inserting {len(chunk_embeddings_rows)} chunk embeddings into {EMBEDDINGS_TABLE_NAME}...")
    
    # Build connection using psycopg2
    conn = psycopg2.connect(
        host=db_host,
        port=db_port,
        dbname=db_name,
        user=db_user,
        password=db_password,
        sslmode='require'
    )
    
    try:
        cursor = conn.cursor()
        
        # Prepare data tuples for batch insert
        # Format embedding as PostgreSQL array literal: '{val1,val2,...}'
        insert_data = [
            (
                row['id'],
                row['alert_id'],
                row['location'],
                int(row['chunk_index']),
                row['chunk_text'],
                '{' + ','.join(str(float(x)) for x in row['embedding']) + '}',
                row['model_name'],
                row['embedded_at']
            )
            for row in chunk_embeddings_rows
        ]
        
        # Batch insert with ON CONFLICT DO NOTHING for deduplication
        insert_sql = f"""
            INSERT INTO {EMBEDDINGS_TABLE_NAME} (
                id, alert_id, location, chunk_index, chunk_text, embedding, model_name, embedded_at
            ) VALUES (%s, %s, %s, %s, %s, %s::double precision[], %s, %s)
            ON CONFLICT (id) DO NOTHING
        """
        
        # executemany in psycopg2 is much faster than individual INSERTs
        cursor.executemany(insert_sql, insert_data)
        
        conn.commit()
        inserted_count = cursor.rowcount
        print(f"✅ Successfully inserted {inserted_count} new chunk embeddings")
        print(f"   (Duplicates were skipped via ON CONFLICT DO NOTHING)")
        print("\nIMPORTANT: Run this SQL in your Lakebase database to cast arrays to vectors:")
        print(f"  UPDATE {EMBEDDINGS_TABLE_NAME} SET embedding = embedding::vector WHERE embedding IS NOT NULL;")
        
    finally:
        cursor.close()
        conn.close()
else:
    print("No chunk embeddings to write.")